# José — Enviar Ruta al Carrito (Laptop)
### Computadora → Carrito vía MQTT

**Cómo usar:**
1. Asegúrate de que `ruta_generada.json` esté en la misma carpeta (lo genera Corzo)
2. Cambia `ROBOT_ID` al número de tu robot el día de la presentación
3. Corre las celdas en orden


In [ ]:
import paho.mqtt.client as mqtt
import json
import time


In [ ]:
# ============================================================
# CONFIGURACIÓN — solo cambia ROBOT_ID el día de la presentación
# ============================================================
BROKER   = "broker.hivemq.com"
PORT     = 1883
ROBOT_ID = "R00"   # ← CAMBIAR al número de su robot

TOPIC_RUTA   = f"{ROBOT_ID}/teleop/ruta"     # Laptop publica aquí → Pico W recibe
TOPIC_STATUS = f"{ROBOT_ID}/sensors/status"  # Pico W publica aquí → Laptop recibe

print(f"Topic envío:       {TOPIC_RUTA}")
print(f"Topic confirmación: {TOPIC_STATUS}")


In [ ]:
# ============================================================
# CALLBACKS MQTT
# ============================================================
carrito_listo = False

def on_connect(client, userdata, flags, rc):
    if rc == 0:
        print(f"[MQTT] Conectado al broker: {BROKER}")
        client.subscribe(TOPIC_STATUS)
        print(f"[MQTT] Escuchando confirmación en: {TOPIC_STATUS}")
    else:
        print(f"[MQTT] Error de conexión. Código: {rc}")

def on_message(client, userdata, msg):
    global carrito_listo
    payload = msg.payload.decode().strip()
    print(f"\n[CARRITO] Mensaje recibido: '{payload}'")
    if payload == "done":
        print("[✓] El carrito terminó el recorrido.")
        carrito_listo = True


In [ ]:
# ============================================================
# LEER RUTA GENERADA POR CORZO
# ============================================================
with open("ruta_generada.json", "r") as f:
    datos = json.load(f)

ruta = datos["ruta"]
print(f"[✓] Ruta cargada: {len(ruta)} puntos")
print(f"    Primer punto: {ruta[0]}")
print(f"    Último punto: {ruta[-1]}")


In [ ]:
# ============================================================
# CONECTAR Y ENVIAR RUTA POR MQTT
# ============================================================
carrito_listo = False

client = mqtt.Client()
client.on_connect = on_connect
client.on_message = on_message

client.connect(BROKER, PORT, keepalive=60)
client.loop_start()
time.sleep(1.5)  # Esperar conexión estable

# Publicar la ruta como JSON
payload = json.dumps({"ruta": ruta})
client.publish(TOPIC_RUTA, payload)

print(f"[→] Ruta enviada al carrito (topic: {TOPIC_RUTA})")
print(f"    {len(payload)} bytes publicados")
print(f"    Primeros 3 puntos: {ruta[:3]}")


In [ ]:
# ============================================================
# ESPERAR CONFIRMACIÓN DEL CARRITO (máx 5 minutos)
# Corre esta celda después de enviar la ruta
# ============================================================
print("[⏳] Esperando que el carrito complete el recorrido...")

timeout = 300  # 5 minutos
t0 = time.time()

while not carrito_listo:
    if time.time() - t0 > timeout:
        print("[TIMEOUT] El carrito no respondió en 5 minutos.")
        break
    time.sleep(0.5)

if carrito_listo:
    print("[✓] Recorrido completado con éxito.")

client.loop_stop()
client.disconnect()
print("[MQTT] Desconectado.")
